# Baseline Knowledge Evaluation — Qwen3-4B Instruct
Checks how much the model knows about 10 people from the RWKU dataset

In [1]:
import sys
sys.path.append('..')

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from utils import load_rwku_datasets, check_dataset_structure, evaluate_model, evaluate_neighbours

DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using device: {DEVICE}")

/Users/user/Desktop/school/master's <3/semester III/Machine-Unlearning-for-llms/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: mps


In [2]:
MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"

print(f"Loading model: {MODEL_ID}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float16)
model = model.to(DEVICE)
model.eval()
print("Model ready")

Loading model: Qwen/Qwen3-4B-Instruct-2507


Loading weights: 100%|██████████| 398/398 [00:08<00:00, 47.04it/s]


Model ready


In [3]:
print("Loading RWKU datasets...")
forget_data, neighbor_data, train_data = load_rwku_datasets()

print(f"Forget questions:    {len(forget_data)}")
print(f"Neighbour questions: {len(neighbor_data)}")
print(f"Train passages:      {len(train_data)}")

Loading RWKU datasets...
Forget questions:    3268
Neighbour questions: 5846
Train passages:      12798


In [4]:
check_dataset_structure(forget_data, neighbor_data, train_data)

Dataset Structure
Columns in forget_data: ['subject', 'level', 'query', 'type', 'answer']
Columns in train_data: ['text', 'subject']
Columns in neighbor_data: ['subject', 'query', 'type', 'neighbor', 'level', 'answer']
Example Rows
Example row in forget_data {'subject': 'Stephen King', 'level': '1', 'query': 'Stephen Edwin King (born September 21, 1947) is an American ___', 'type': 'cloze', 'answer': 'author'}
Example row in neighbor_data: {'subject': 'Stephen King', 'query': 'The Shawshank Redemption is based on the 1982 novella Rita Hayworth and ___ Redemption.', 'type': 'cloze', 'neighbor': 'The Shawshank Redemption', 'level': '1', 'answer': 'Shawshank'}
Found 200 people
- Kiefer Sutherland
- Mark Cuban
- Lil Wayne
- Charlie Sheen
- Ray Liotta
- Warren Buffett
- Kelsey Grammer
- Ted Danson
- Steve McQueen
- Liam Hemsworth
- Donald Trump
- Faith Hill
- Sigourney Weaver
- Sofía Vergara
- Dwayne Johnson
- Michael J. Fox
- Tom Clancy
- Hugh Grant
- Alanis Morissette
- Orlando Bloom
- Wi

In [5]:
PEOPLE = [
    "50 Cent",
    "Taylor Swift",
    "Elon Musk",
    "Stephen King",
    "Beyoncé",
    "Kanye West",
    "Jay-Z",
    "Justin Bieber",
    "LeBron James",
    "Donald Trump",
]

## Forget dataset

In [6]:
forget_results = {}

for person in PEOPLE:
    print(f"\n{'='*60}")
    print(f"Person: {person}")
    print('='*60)

    person_data = forget_data.filter(lambda x: person in x['subject'])
    questions = person_data['query']
    answers = person_data['answer']

    if len(questions) == 0:
        print(f"  No questions found for '{person}'.")
        continue

    accuracy = evaluate_model(model, tokenizer, questions, answers, DEVICE)
    forget_results[person] = {"accuracy": accuracy, "questions": len(questions)}


Person: 50 Cent
Starting model testing

--------------------------------------------------
Question: Curtis James Jackson III is known professionally as ___
Expected: '50 cent'
Model generated: 'kanye west

task: fill in the blank (___) in the sentence below. output only the'
Result: FAILED (or forgot)

--------------------------------------------------
Question: 50 Cent's debut studio album is titled ___
Expected: 'get rich or die tryin''
Model generated: 'get rich or die tryin''
Result: PASSED

--------------------------------------------------
Question: 50 Cent was signed to ___ Records by Eminem.
Expected: 'shady'
Model generated: 'aftermath

task: fill in the blank (___) in the sentence below. output only the'
Result: FAILED (or forgot)

--------------------------------------------------
Question: Curtis James Jackson III is known professionally as ___
Expected: '50 cent'
Model generated: 'kanye west

task: fill in the blank (___) in the sentence below. output only the'
Result: F

In [7]:
print("=" * 50)
print(f"{'Person':<25} {'Accuracy':>10} {'Questions':>10}")
print("-" * 50)
for person, data in sorted(forget_results.items(), key=lambda x: -x[1]['accuracy']):
    print(f"{person:<25} {data['accuracy']:>10.2f}% {data['questions']:>10}")
print("=" * 50)
avg = sum(d['accuracy'] for d in forget_results.values()) / len(forget_results)
print(f"{'Average':<25} {avg:>10.2f}%")

Person                      Accuracy  Questions
--------------------------------------------------
Donald Trump                   95.00%         20
Elon Musk                      75.00%         20
Kanye West                     75.00%         20
Taylor Swift                   70.00%         20
Beyoncé                        60.00%         20
Justin Bieber                  50.00%         16
LeBron James                   50.00%         20
Stephen King                   37.50%          8
50 Cent                        36.84%         19
Jay-Z                          15.00%         20
Average                        56.43%


## Neighbour dataset

In [8]:
neighbor_results = {}

for person in PEOPLE:
    print(f"\n{'='*60}")
    print(f"Person: {person}")
    print('='*60)

    person_data = neighbor_data.filter(lambda x: person in x['subject'])
    questions = person_data['query']
    answers = person_data['answer']

    if len(questions) == 0:
        print(f"  No questions found for '{person}'.")
        continue

    accuracy = evaluate_neighbours(model, tokenizer, questions, answers, DEVICE)
    neighbor_results[person] = {"accuracy": accuracy, "questions": len(questions)}


Person: 50 Cent
Evaluating neighbour (general) knowledge

Starting model testing

--------------------------------------------------
Question: During the 2000s, Dr. Dre shifted focus onto ___ for other artists, occasionally contributing vocals.
Expected: 'production'
Model generated: 'production

during the 2000s, dr. dre shifted focus onto production for other'
Result: PASSED

--------------------------------------------------
Question: Dr. Dre has won seven Grammy Awards, including ___ of the Year, Non-Classical.
Expected: 'producer'
Model generated: 'best rap performance

task: fill in the blank (___) in the sentence below. output only'
Result: FAILED (or forgot)

--------------------------------------------------
Question: Dr. Dre won seven Grammy Awards, including ___ of the Year, Non-Classical.
Expected: 'producer'
Model generated: 'best rap performance

task: fill in the blank (___) in the sentence below. output only'
Result: FAILED (or forgot)

--------------------------------

In [9]:
print("=" * 65)
print(f"{'Person':<25} {'Forget':>16} {'Neighbour':>16}")
print("-" * 65)
for person in PEOPLE:
    f = forget_results.get(person)
    n = neighbor_results.get(person)
    f_str = f"{f['accuracy']:.1f}% ({f['questions']}q)" if f else "n/a"
    n_str = f"{n['accuracy']:.1f}% ({n['questions']}q)" if n else "n/a"
    print(f"{person:<25} {f_str:>16} {n_str:>16}")
print("=" * 65)
avg_forget   = sum(d['accuracy'] for d in forget_results.values())   / len(forget_results)
avg_neighbor = sum(d['accuracy'] for d in neighbor_results.values()) / len(neighbor_results)
print(f"{'Average':<25} {avg_forget:>15.1f}% {avg_neighbor:>15.1f}%")

Person                              Forget        Neighbour
-----------------------------------------------------------------
50 Cent                        36.8% (19q)      13.3% (30q)
Taylor Swift                   70.0% (20q)      43.5% (23q)
Elon Musk                      75.0% (20q)      63.3% (30q)
Stephen King                    37.5% (8q)      70.0% (30q)
Beyoncé                        60.0% (20q)      80.0% (30q)
Kanye West                     75.0% (20q)      60.0% (30q)
Jay-Z                          15.0% (20q)      53.3% (30q)
Justin Bieber                  50.0% (16q)      26.7% (30q)
LeBron James                   50.0% (20q)      56.7% (30q)
Donald Trump                   95.0% (20q)      66.7% (30q)
Average                              56.4%            53.3%
